# 리뷰 한 건 따라가기

새 리뷰 한 건이 들어오면 어떤 순서로 처리되는지, 단계마다 실제 출력을 보면서 따라갑니다.

```
원문 리뷰
 → ① 전처리 (normalize_text: HTML 기호, 공백 정리)
 → ② Kiwi: 형태소 분석 → 문장 나누기 → 연결어미에서 절 나누기 → 토막 합치기  = "단위"
 → ③ KcELECTRA(식품 리뷰로 사후학습 + 단위 감성 미세조정): 단위마다 긍정 / 부정 / 평가 아님
 → ④ KURE-v1: 단위 문장을 1,024차원 벡터로 (뜻이 비슷하면 가까움)
 → ⑤ BERTopic: 상품의 같은 감성 단위들을 UMAP → HDBSCAN으로 묶고, c-TF-IDF로 키워드
 → ⑥ 새 단위를 가장 가까운 묶음에 배정 (유사도 0.75 이상일 때만)
 → ⑦ 집계: 묶음별 건수, 최근 3개월 vs 그 이전 (리뷰 100건당)
```

예시 상품은 **노르웨이고등어**(상품 ID `300868056`, 리뷰 2만 건)입니다. ⑤의 묶음은 이 상품의 실제 리뷰로 이미 만들어 둔 결과(`6_이유분석/results-v3`)를 씁니다.

In [1]:
import os, sys, json, re
os.environ.update(USE_TF='0', TRANSFORMERS_NO_TF='1', TOKENIZERS_PARALLELISM='false')
sys.modules['tensorflow'] = None          # umap이 쓰지도 않는 TensorFlow를 불러오지 않게
from pathlib import Path
# 프로젝트 맨 위 폴더 찾기: VS Code는 노트북이 있는 폴더(7_보고/노트북)에서 실행하므로 위로 올라가며 찾음
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'src' / 'bibi_reviews').is_dir())
sys.path[:0] = [str(ROOT / 'src'), str(ROOT / '5_감성분류/단위나누기'), str(ROOT / '5_감성분류/분류기'), str(ROOT / '6_이유분석')]

import bibi_reviews                       # 불러오는 순간 이 PC의 효율 코어에만 고정됨 (bibi_reviews/cpu.py)
from bibi_reviews.paths import KURE_V1
from bibi_reviews.backbone import normalize_text
import numpy as np, pandas as pd, torch
pd.set_option('display.max_colwidth', 80)
PRODUCT = '300868056'
RESULTS = ROOT / '6_이유분석/results-v3' / PRODUCT
print('준비 완료 | GPU:', torch.cuda.get_device_name(0))

준비 완료 | GPU: NVIDIA GeForce RTX 4080 SUPER


## 0. 새 리뷰

실제 리뷰처럼 칭찬과 불만이 섞인 문장입니다. 일부러 `&hellip;`(말줄임표 HTML 기호)와 겹친 공백을 넣었습니다. 원본 CSV에는 이런 기호가 섞여 들어옵니다.

In [2]:
raw = "가시가 없어서 아이들 먹이기 편해요&hellip; 그런데 이번엔 크기가 좀 작고   살이 퍽퍽했어요. 비린내는 거의 안 나요! 배송은 빨랐는데 아이스팩이 다 녹아서 왔네요ㅠ 다음엔 특대로 시켜볼게요"
print(raw)

가시가 없어서 아이들 먹이기 편해요&hellip; 그런데 이번엔 크기가 좀 작고   살이 퍽퍽했어요. 비린내는 거의 안 나요! 배송은 빨랐는데 아이스팩이 다 녹아서 왔네요ㅠ 다음엔 특대로 시켜볼게요


## ① 전처리

`normalize_text`는 HTML 기호를 원래 문자로 되돌리고 공백을 정리합니다. **원문의 뜻은 바꾸지 않습니다**(오타를 고치지 않음). 결과의 근거 문장이 실제 리뷰와 같아야 하기 때문입니다.

In [3]:
text = normalize_text(raw)
print('전:', raw)
print('후:', text)

전: 가시가 없어서 아이들 먹이기 편해요&hellip; 그런데 이번엔 크기가 좀 작고   살이 퍽퍽했어요. 비린내는 거의 안 나요! 배송은 빨랐는데 아이스팩이 다 녹아서 왔네요ㅠ 다음엔 특대로 시켜볼게요
후: 가시가 없어서 아이들 먹이기 편해요… 그런데 이번엔 크기가 좀 작고 살이 퍽퍽했어요. 비린내는 거의 안 나요! 배송은 빨랐는데 아이스팩이 다 녹아서 왔네요ㅠ 다음엔 특대로 시켜볼게요


## ② Kiwi로 단위 나누기

### ②-1 형태소 분석
Kiwi는 문장을 형태소와 품사로 나눕니다. 단위를 자르는 기준이 되는 품사는 `EC`(연결어미: ~는데, ~지만, ~고, ~면서)입니다.

In [4]:
from kiwipiepy import Kiwi
kiwi = Kiwi()
toks = kiwi.tokenize(text)
pd.DataFrame([(t.form, t.tag, t.start) for t in toks], columns=['형태소', '품사', '위치']).T

,0,1,2,3,4,5,6,7,8,9,...,48,49,50,51,52,53,54,55,56,57
형태소,가시,가,없,어서,아이,들,먹이,기,편하,어요,...,ㅠ,다음,에,ᆫ,특,대로,시키,어,보,ᆯ게요
품사,NNG,JKS,VA,EC,NNG,XSN,VV,ETN,VA,EF,...,SW,NNG,JKB,JX,NNG,JX,VV,EC,VX,EF
위치,0,2,4,5,8,10,12,14,16,17,...,86,88,90,90,92,93,96,97,98,98


### ②-2 문장 나누기 → 절 나누기 → 토막 합치기
1. 문장 단위로 나눕니다.
2. 문장 안에서 `-는데`, `-지만`, `-고`, `-면서` 같은 연결어미 뒤를 자릅니다. 단, 뒤에 보조동사가 오면('먹고 **있어요**') 자르지 않습니다.
3. 서술어 없이 조사로 끝나는 토막('옆지느러미가.')은 다음 단위에 붙입니다.

In [5]:
from kiwi_units import units, is_clause_end, CLAUSE_EC
for s in kiwi.split_into_sents(text):
    sent_toks = kiwi.tokenize(s.text)
    cuts = [t.form for i, t in enumerate(sent_toks) if is_clause_end(sent_toks, i, CLAUSE_EC)]
    print(f'문장: {s.text}\n   자르는 곳(연결어미): {cuts}')
spans = units(kiwi, text, True)
unit_texts = [text[s:e] for s, e in spans]
pd.DataFrame({'단위': unit_texts, '원문 위치': [f'{s}~{e}' for s, e in spans]})

문장: 가시가 없어서 아이들 먹이기 편해요…
   자르는 곳(연결어미): []
문장: 그런데 이번엔 크기가 좀 작고 살이 퍽퍽했어요.
   자르는 곳(연결어미): ['고']
문장: 비린내는 거의 안 나요!
   자르는 곳(연결어미): []
문장: 배송은 빨랐는데 아이스팩이 다 녹아서 왔네요ㅠ
   자르는 곳(연결어미): ['데']
문장: 다음엔 특대로 시켜볼게요
   자르는 곳(연결어미): []


,단위,원문 위치
0,가시가 없어서 아이들 먹이기 편해요…,0~20
1,그런데 이번엔 크기가 좀 작고,21~37
2,살이 퍽퍽했어요.,38~47
3,비린내는 거의 안 나요!,48~61
4,배송은 빨랐는데,62~70
5,아이스팩이 다 녹아서 왔네요ㅠ,71~87
6,다음엔 특대로 시켜볼게요,88~101


## ③ KcELECTRA: 단위마다 감성 판정

- 바탕 모델은 `beomi/KcELECTRA-base-v2022`(한국어 댓글로 사전학습)입니다. 이 모델을 우리 리뷰 49.5만 건으로 **사후학습**해 식품 리뷰 표현에 익숙하게 만들었어요(`3_사후학습/frozen-encoder`).
- 그 위에 **감성 분류 미세조정**을 했어요. 정답 300건에서 나온 단위와, 30개 상품에서 뽑아 사람이 확인한 단위 799개를 썼습니다(`6_이유분석/models/unit-sentiment-v3`).
- 확신도가 0.8보다 낮으면 "재확인"으로 따로 모읍니다.

### ③-1 토큰화
KcELECTRA는 Kiwi와 별개로 자기 사전(WordPiece)으로 글자를 쪼개서 읽습니다.

In [6]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
MODEL = ROOT / '6_이유분석/models/unit-sentiment-v3'
tokenizer = AutoTokenizer.from_pretrained(str(MODEL), local_files_only=True)
clf = AutoModelForSequenceClassification.from_pretrained(str(MODEL), local_files_only=True).cuda().eval()
for u in unit_texts[:3]:
    print(f'{u}\n   → {tokenizer.tokenize(u)}')

C:\Users\MingyunKang\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 14989.78it/s]

가시가 없어서 아이들 먹이기 편해요…
   → ['가시', '##가', '없어서', '아이들', '먹이', '##기', '편해', '##요', '…']
그런데 이번엔 크기가 좀 작고
   → ['그런데', '이번엔', '크기', '##가', '좀', '작', '##고']
살이 퍽퍽했어요.
   → ['살이', '퍽', '##퍽', '##했어요', '.']


In [7]:
LABELS = {0: '긍정', 1: '부정', 2: '평가 아님'}
enc = tokenizer(unit_texts, truncation=True, max_length=128, padding=True, return_tensors='pt').to('cuda')
with torch.inference_mode():
    prob = torch.softmax(clf(**enc).logits.float(), -1).cpu().numpy()
judged = pd.DataFrame(prob.round(3), columns=['긍정', '부정', '평가 아님'])
judged.insert(0, '단위', unit_texts)
judged['판정'] = [LABELS[i] if p.max() >= 0.8 else f'재확인({LABELS[i]}?)' for i, p in zip(prob.argmax(1), prob)]
judged['확신도'] = prob.max(1).round(3)
judged

,단위,긍정,부정,평가 아님,판정,확신도
0,가시가 없어서 아이들 먹이기 편해요…,0.992,0.003,0.004,긍정,0.992
1,그런데 이번엔 크기가 좀 작고,0.002,0.993,0.005,부정,0.993
2,살이 퍽퍽했어요.,0.002,0.993,0.005,부정,0.993
3,비린내는 거의 안 나요!,0.987,0.008,0.005,긍정,0.987
4,배송은 빨랐는데,0.961,0.005,0.034,긍정,0.961
5,아이스팩이 다 녹아서 왔네요ㅠ,0.003,0.993,0.005,부정,0.993
6,다음엔 특대로 시켜볼게요,0.008,0.008,0.984,평가 아님,0.984


## ④ KURE-v1: 문장을 벡터로

KURE-v1은 문장의 **뜻**을 1,024개 숫자로 바꿉니다. 뜻이 비슷한 문장은 벡터가 가깝습니다(코사인 유사도 1에 가까움). 아래 표는 단위끼리의 유사도입니다.

In [8]:
from sentence_transformers import SentenceTransformer
kure = SentenceTransformer(str(KURE_V1), device='cuda', local_files_only=True)
vec = kure.encode(unit_texts, normalize_embeddings=True)
print('벡터 크기:', vec.shape)
pd.DataFrame((vec @ vec.T).round(2), index=unit_texts, columns=[u[:8] for u in unit_texts]).style.background_gradient(cmap='Blues', axis=None)

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 391/391 [00:00<00:00, 20686.88it/s]

벡터 크기: (7, 1024)


,가시가 없어서,그런데 이번엔,살이 퍽퍽했어요,비린내는 거의,배송은 빨랐는데,아이스팩이 다,다음엔 특대로
가시가 없어서 아이들 먹이기 편해요…,1.000000,0.450000,0.460000,0.540000,0.480000,0.430000,0.400000
그런데 이번엔 크기가 좀 작고,0.450000,1.000000,0.490000,0.440000,0.550000,0.460000,0.530000
살이 퍽퍽했어요.,0.460000,0.490000,1.000000,0.480000,0.550000,0.570000,0.470000
비린내는 거의 안 나요!,0.540000,0.440000,0.480000,1.000000,0.450000,0.440000,0.390000
배송은 빨랐는데,0.480000,0.550000,0.550000,0.450000,1.000000,0.580000,0.530000
아이스팩이 다 녹아서 왔네요ㅠ,0.430000,0.460000,0.570000,0.440000,0.580000,1.000000,0.470000
다음엔 특대로 시켜볼게요,0.400000,0.530000,0.470000,0.390000,0.530000,0.470000,1.000000


## ⑤ BERTopic: 상품의 단위들은 어떻게 묶였나

상품 하나의 **같은 감성** 단위를 모아 묶습니다(불만은 불만끼리, 칭찬은 칭찬끼리).

1. **KURE 벡터** (④와 같음)
2. **UMAP**: 1,024차원을 5차원으로 줄여 가까운 것끼리 더 잘 보이게 합니다.
3. **HDBSCAN**: 빽빽하게 모인 곳을 묶음으로 잡고, 어디에도 안 맞는 문장은 "묶이지 않음"(-1)으로 둡니다.
4. **c-TF-IDF**(Kiwi 형태소 기준): 묶음마다 특징적인 단어를 키워드로 뽑습니다.
5. **같은 뜻 묶음 합치기**: 묶음끼리 대표 문장 유사도가 0.78 이상이면 합칩니다(사람이 판정한 148쌍으로 정한 경계). 구체적인 묶음과 일반 묶음은 합치지 않습니다.
6. **소속 확인**: 문장마다 가장 가까운 묶음 중심(묶음 문장 벡터의 평균)과 유사도가 0.75 이상이면 그 묶음, 아니면 "묶이지 않음"입니다.

아래는 고등어 불만 단위로 1~4를 그 자리에서 다시 돌린 결과입니다(1~2분).

In [9]:
from collections import Counter
from bertopic import BERTopic
from umap import UMAP
from hdbscan import HDBSCAN
from sklearn.feature_extraction.text import CountVectorizer

rows = [json.loads(l) for l in (RESULTS / 'units.jsonl').read_text(encoding='utf-8').splitlines()]
neg = Counter(re.sub(r'\s+', ' ', r['unit']).strip() for r in rows if r['label'] == 'NEG')
docs = list(neg)
emb = kure.encode(docs, batch_size=128, normalize_embeddings=True, show_progress_bar=False)
keep = {'NNG', 'NNP', 'VV', 'VA', 'XR', 'MAG'}
words = lambda s: [t.form + ('다' if t.tag in ('VV', 'VA') else '') for t in kiwi.tokenize(s) if t.tag in keep]
# embedding_model을 꼭 넘길 것: 안 넘기면 BERTopic 기본 언어가 'english'라서 키워드를 뽑기 전에 한글을 전부 지워 버림
topic_model = BERTopic(embedding_model=kure, umap_model=UMAP(n_neighbors=15, n_components=5, min_dist=0.0, metric='cosine', random_state=20261022),
                       hdbscan_model=HDBSCAN(min_cluster_size=max(15, round(0.01 * len(docs))), prediction_data=True),
                       vectorizer_model=CountVectorizer(tokenizer=words, lowercase=False, min_df=2, token_pattern=None),
                       top_n_words=8)
topics, _ = topic_model.fit_transform(docs, emb)
print(f'불만 문장 {len(docs):,}개(서로 다른 문장) → 묶음 {len(set(topics)) - 1}개, 묶이지 않음 {np.mean(np.array(topics) == -1):.0%}')
info = topic_model.get_topic_info()
info['키워드'] = info['Topic'].map(lambda t: '·'.join(w for w, _ in topic_model.get_topic(t)[:5]) if t != -1 else '(묶이지 않음)')
info['예시'] = info['Representative_Docs'].map(lambda d: ' / '.join(d[:2]))
info[['Topic', 'Count', '키워드', '예시']].head(12)

불만 문장 2,467개(서로 다른 문장) → 묶음 26개, 묶이지 않음 20%


,Topic,Count,키워드,예시
0,-1,494,(묶이지 않음),다만 크기 차이가 조금 있어요~~ / 조금 더 크면 좋겠지만
1,0,204,맛·그냥·무난·덜·없다,그치만 맛은 고트입니다. / 얕은맛이 잇어요
2,1,186,간·소금·되다·안·뿌리다,소금간이하나도안되어있음 / 소금간은 안되어 있다고 아들이 그러네요.
3,2,182,고등어·오르다·맛·노르웨이·값,고등어가 이게 뭡니까.. / 고등어가 두껍고
4,3,177,작다·크기·사이즈·좀·조금,크기가 좀 작긴한데 / 크기도 좀 작고
5,4,114,비린내·나다·비리다·살짝·냄새,비린내도 나고 / 구운 뒤 비린내 나서
6,5,99,크다·특대·생각·대·특,특대가 생각보다 크진 않네요~ / 특대라는데 생각만큼 크진 않네요.
7,6,95,가시·나오다·있다·가끔·잘다,가시는 몇가닥있었지만 / 가시있음
8,7,93,생각·작다·크기·사이즈·기대,생각보다는 작네요 / 생각보다 작긴한데
9,8,75,비싸다·가격·좀·싸다·조금,가격이 좀 비싸요 / 가격이 좀 비싸져서 아쉽지만


실제 분석(`run_product.py`)은 여기에 합치기(5)와 소속 확인(6)까지 한 결과를 저장해 둡니다. 저장된 고등어 불만 묶음:

In [10]:
report = json.loads((RESULTS / 'report.json').read_text(encoding='utf-8'))
saved = pd.DataFrame([{'묶음': g['topic'], '건수': g['units'], '키워드': '·'.join(g['keywords'][:4]),
                       '대표 문장': (g.get('representative') or [''])[0], '합친 묶음 수': len(g.get('merged_from', [g['topic']]))}
                      for g in report['groups']['NEG']['groups'] if g['topic'] != -1])
saved.head(10)

,묶음,건수,키워드,대표 문장,합친 묶음 수
0,3,430,작다·크기·사이즈·좀,크기는 조금 작은데,3
1,2,172,고등어·오르다·맛·노르웨이,고등어가 작긴하네요,1
2,1,153,간·소금·되다·안,간이 안되어 있어서 소금은 좀 뿌려야 해요,1
3,0,137,맛·그냥·무난·덜,맛은 그럭저럭괟찮아요,1
4,4,95,비린내·나다·비리다·살짝,비린내가 좀 많이 나요,1
5,8,84,비싸다·가격·좀·싸다,가격이 조금비싸요,1
6,6,81,가시·나오다·있다·가끔,가시가 조금 있긴한데,1
7,11,70,단백·입맛·싱싱·약간,살짝 싱거운듯한데,1
8,10,67,녹다·오다·다·살짝,좀 녹아서 왔네요.,1
9,18,65,실망·심심하다·짜다·속상하다,그게 좀 아쉬워요ㅠ,1


## ⑥ 새 단위를 기존 묶음에 배정

새 리뷰가 들어올 때마다 묶음 전체를 다시 만들지 않습니다. 새 단위를 **같은 감성**의 기존 묶음과 비교해서, 가장 가까운 묶음의 대표 문장과 유사도가 0.75 이상이면 그 묶음에 넣습니다. 0.75보다 낮으면 "새 의견 후보"로 남깁니다. 이런 후보가 쌓이면 다음 정기 실행 때 새 묶음이 생길 수 있어요.

In [11]:
MEMBER_MIN = 0.75
KEY = {'긍정': 'POS', '부정': 'NEG'}
group_vecs = {}
for label in ['NEG', 'POS']:
    for g in report['groups'][label]['groups']:
        if g['topic'] != -1 and g.get('representative'):
            group_vecs[(label, g['topic'])] = (g, kure.encode(g['representative'], normalize_embeddings=True))
out = []
for u, v, verdict in zip(unit_texts, vec, judged['판정']):
    label = KEY.get(verdict)
    if label is None:
        out.append((u, verdict, '-', None, '감성 평가가 아니라 묶지 않음'))
        continue
    best = max(((g, float((gv @ v).max())) for (l, _), (g, gv) in group_vecs.items() if l == label), key=lambda x: x[1])
    g, sim = best
    name = f"[{'·'.join(g['keywords'][:3])}] {g['representative'][0]}"
    out.append((u, verdict, name, round(sim, 3), '배정' if sim >= MEMBER_MIN else '새 의견 후보'))
assigned = pd.DataFrame(out, columns=['단위', '판정', '가장 가까운 묶음', '유사도', '결과'])
assigned

,단위,판정,가장 가까운 묶음,유사도,결과
0,가시가 없어서 아이들 먹이기 편해요…,긍정,[가시·없다·편하다] 가시 없어서 먹기 편하네요,0.896,배정
1,그런데 이번엔 크기가 좀 작고,부정,[작다·크기·사이즈] 크기는 조금 작은데,0.831,배정
2,살이 퍽퍽했어요.,부정,[퍽퍽하다·살·퍽퍽] 근데 약간 퍽퍽해요.,0.795,배정
3,비린내는 거의 안 나요!,긍정,"[비린내·안·간] , 비린내도 안나고",0.812,배정
4,배송은 빨랐는데,긍정,[포장·배송·빠르다] 배송이 빠르네요.,0.878,배정
5,아이스팩이 다 녹아서 왔네요ㅠ,부정,[아이스·팩·추가] 추가해도 아이스팩이 다 녹아있더라고요.,0.884,배정
6,다음엔 특대로 시켜볼게요,평가 아님,-,NaN,감성 평가가 아니라 묶지 않음


## ⑦ 집계

배정된 단위는 그 묶음의 건수에 더해집니다. 보고서는 묶음마다 **최근 3개월**과 **그 이전**의 리뷰 100건당 건수를 비교해서, 최근에 늘어난 불만을 찾습니다. 일반적인 칭찬·불만('맛있어요', '그냥 그래요')은 건수에는 남기고 보고서 앞에는 두지 않습니다.

In [12]:
months, recent = report['period'], report['recent_months']
print(f"기간 {months[0]} ~ {months[1]} | 최근 3개월 {recent} | 최근 리뷰 {report['recent_reviews']:,}건 / 전체 {report['reviews']:,}건")
hit = [a for a in out if a[4] == '배정' and a[1] == '부정']
rows_ = []
for g in report['groups']['NEG']['groups']:
    if g['topic'] == -1:
        continue
    name = f"[{'·'.join(g['keywords'][:3])}] {g['representative'][0]}"
    if any(h[2] == name for h in hit):
        rows_.append({'불만 묶음': name, '지금까지 건수': g['units'], '이번 리뷰로 +': sum(h[2] == name for h in hit),
                      '이전 (100건당)': g['per_100_reviews_earlier'], '최근 (100건당)': g['per_100_reviews_recent']})
pd.DataFrame(rows_)

기간 2025-01 ~ 2026-10 | 최근 3개월 ['2026-08', '2026-09', '2026-10'] | 최근 리뷰 1,804건 / 전체 20,565건


,불만 묶음,지금까지 건수,이번 리뷰로 +,이전 (100건당),최근 (100건당)
0,[작다·크기·사이즈] 크기는 조금 작은데,430,1,2.06,2.49
1,[아이스·팩·추가] 추가해도 아이스팩이 다 녹아있더라고요.,44,1,0.18,0.55
2,[퍽퍽하다·살·퍽퍽] 근데 약간 퍽퍽해요.,40,1,0.21,0.00


## 정리

| 단계 | 도구 | 이 리뷰에서 한 일 |
| --- | --- | --- |
| ① 전처리 | `normalize_text` | `&hellip;` → `…`, 겹친 공백 정리 |
| ② 단위 나누기 | Kiwi | 한 리뷰를 의견 단위 여러 개로 자름 |
| ③ 감성 | KcELECTRA (사후학습 + 미세조정) | 단위마다 긍정 / 부정 / 평가 아님과 확신도 |
| ④ 의미 벡터 | KURE-v1 | 뜻이 비슷한 단위는 유사도가 높음 |
| ⑤ 묶기 | BERTopic (UMAP → HDBSCAN → c-TF-IDF) + 합치기 + 소속 확인 | 상품 전체 단위로 묶음과 키워드를 만듦 |
| ⑥ 배정 | KURE 유사도 ≥ 0.75 | 새 단위를 기존 묶음에 넣거나 새 의견 후보로 남김 |
| ⑦ 집계 | 리뷰 100건당 건수 | 묶음별 건수, 최근 3개월 vs 이전 |

이 노트북은 `7_보고/노트북/make_notebook.py`로 다시 만들 수 있습니다. 0번 셀의 `raw`를 바꾸면 다른 리뷰로도 따라가 볼 수 있습니다.